In [ ]:
import os, random, math, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

# Config
SEED = 42
BATCH_TRAIN = 32
BATCH_VAL = 32
EPOCHS = 20
LR = 2e-5
MAX_LEN = 512
MODEL_NAME = "elmurod1202/bertbek-news-big-cased"

device = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
import numpy as np 
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "/kaggle/input/kunuz-cleaned/kunuz_final.csv"
df = pd.read_csv(file_path)

df = df[df['gender'] != 'unknown']
# not 17620 
len(df)

In [ ]:
from transformers import DataCollatorWithPadding
from sklearn.preprocessing import LabelEncoder

le_gender = LabelEncoder()
le_age = LabelEncoder()
le_industry = LabelEncoder()

df["gender_lbl"]   = le_gender.fit_transform(df["gender"])
df["year_lbl"] = le_age.fit_transform(df["year"])
df["topic_lbl"]  = le_industry.fit_transform(df["category"])

class MTDataset(Dataset):
    def __init__(self, df, max_len=MAX_LEN):
        self.texts = df["body"].astype(str).tolist()
        self.g = df["gender_lbl"].astype(int).tolist()
        self.y = df["year_lbl"].astype(int).tolist()
        self.t = df["topic_lbl"].astype(int).tolist()
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = tok(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,   
            return_tensors="pt"        
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["g"] = torch.tensor(self.g[idx], dtype=torch.long)
        item["y"] = torch.tensor(self.y[idx], dtype=torch.long)
        item["t"] = torch.tensor(self.t[idx], dtype=torch.long)
        return item

collate_fn = DataCollatorWithPadding(tokenizer=tok, return_tensors="pt")

train_df, val_df = train_test_split(
    df, test_size=0.15, random_state=SEED, stratify=df["gender_lbl"]
)

ds_tr, ds_va = MTDataset(train_df, max_len=MAX_LEN), MTDataset(val_df, max_len=MAX_LEN)

dl_tr = DataLoader(
    ds_tr, batch_size=BATCH_TRAIN, shuffle=True,
    collate_fn=collate_fn, num_workers=4, pin_memory=True, persistent_workers=True
)
dl_va = DataLoader(
    ds_va, batch_size=BATCH_VAL, shuffle=False,
    collate_fn=collate_fn, num_workers=4, pin_memory=True, persistent_workers=True
)

num_year = df["year_lbl"].nunique()
num_topic = df["topic_lbl"].nunique()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

class MTLModel(nn.Module):
    def __init__(self, base, n_year, n_topic, use_last4_weighted=True):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base)
        hid = self.encoder.config.hidden_size

        self.use_last4_weighted = use_last4_weighted

        if self.use_last4_weighted:
            self.layer_weights = nn.Parameter(torch.zeros(4))  

        self.dropout = nn.Dropout(0.1)

        self.head_g = nn.Linear(hid, 2)
        self.head_y = nn.Linear(hid, n_year)
        self.head_t = nn.Linear(hid, n_topic)

        self.ce_g = nn.CrossEntropyLoss(label_smoothing=0.05)
        self.ce_y = nn.CrossEntropyLoss(label_smoothing=0.05)
        self.ce_t = nn.CrossEntropyLoss(label_smoothing=0.05)

    def _pool_cls(self, layer_hidden):
        return layer_hidden[:, 0, :]  

    def forward(self, input_ids, attention_mask, g=None, y=None, t=None):
        if self.use_last4_weighted:
            out = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                return_dict=True
            )
            hs = out.hidden_states  

            last4 = [hs[-1], hs[-2], hs[-3], hs[-4]]  
            reps = [self._pool_cls(x) for x in last4]  

            # weighted sum
            alpha = F.softmax(self.layer_weights, dim=0)  
            h = alpha[0]*reps[0] + alpha[1]*reps[1] + alpha[2]*reps[2] + alpha[3]*reps[3]
        else:
            out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
            h = self._pool_cls(out.last_hidden_state)

        h = self.dropout(h)

        lg = self.head_g(h)  # (B,2)
        ly = self.head_y(h)  # (B,n_year)
        lt = self.head_t(h)  # (B,n_topic)

        losses = None
        if (g is not None) and (y is not None) and (t is not None):
            Lg = self.ce_g(lg, g)
            Ly = self.ce_y(ly, y)
            Lt = self.ce_t(lt, t)
            loss = Lg + Ly + Lt
            losses = (loss, Lg.detach(), Ly.detach(), Lt.detach())

        return (lg, ly, lt), losses


model = MTLModel(MODEL_NAME, num_year, num_topic).to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
num_train_steps = EPOCHS * len(dl_tr)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=num_train_steps)

def evaluate():
    model.eval()
    g_pred, g_true, y_pred, y_true, t_pred, t_true = [], [], [], [], [], []

    with torch.no_grad():
        for batch in dl_va:
            for k in ("input_ids","attention_mask","g","y","t"):
                batch[k] = batch[k].to(device)

            (lg, ly, lt), _ = model(batch["input_ids"], batch["attention_mask"])

            g_pred.extend(lg.argmax(1).cpu().numpy())
            g_true.extend(batch["g"].cpu().numpy())

            y_pred.extend(ly.argmax(1).cpu().numpy())
            y_true.extend(batch["y"].cpu().numpy())

            t_pred.extend(lt.argmax(1).cpu().numpy())
            t_true.extend(batch["t"].cpu().numpy())

    return {
        "G_acc": accuracy_score(g_true, g_pred),
        "G_f1": f1_score(g_true, g_pred, average="macro"),
        "Y_acc": accuracy_score(y_true, y_pred),
        "Y_f1": f1_score(y_true, y_pred, average="macro"),
        "T_acc": accuracy_score(t_true, t_pred),
        "T_f1": f1_score(t_true, t_pred, average="macro"),
    }

In [ ]:
from tqdm.auto import tqdm
import torch

CKPT_DIR = "/kaggle/working/bertbek_mtl"
os.makedirs(CKPT_DIR, exist_ok=True)

best_comp = -1.0
best_epoch = -1
best_path = os.path.join(CKPT_DIR, "best.pt")

for ep in range(1, EPOCHS + 1):
    model.train()
    tr_losses = []
    pbar = tqdm(dl_tr, desc=f"Epoch {ep}", leave=False)

    for batch in pbar:
        for k in ("input_ids", "attention_mask", "g", "y", "t"):
            batch[k] = batch[k].to(device)
            
        (_, _, _), losses = model(
            batch["input_ids"],
            batch["attention_mask"],
            batch["g"],
            batch["y"],
            batch["t"],
        )

        loss = losses[0]
        if loss.dim() > 0:  
            loss = loss.mean()

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        tr_losses.append(loss.item())
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    # evaluation
    metrics = evaluate()
    comp = (metrics["G_f1"] + metrics["Y_f1"] + metrics["T_f1"]) / 3.0

    print(
        f"Epoch {ep:02d} | train {np.mean(tr_losses):.4f} | "
        f"G {metrics['G_acc']:.3f}/{metrics['G_f1']:.3f} | "
        f"Y {metrics['Y_acc']:.3f}/{metrics['Y_f1']:.3f} | "
        f"T {metrics['T_acc']:.3f}/{metrics['T_f1']:.3f} | comp {comp:.3f}"
    )

    if comp > best_comp:
        best_comp = comp
        best_epoch = ep

        state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
        torch.save(
            {
                "epoch": ep,
                "best_comp": best_comp,
                "state_dict": state,
            },
            best_path
        )
        print(f"✅ New best! Saved: {best_path} (epoch {ep}, comp {best_comp:.3f})")

print(f"\nBest epoch = {best_epoch}, best comp = {best_comp:.3f}")
print(f"Best checkpoint: {best_path}")